# Eve demo: a calculator

Define tools as plain Python functions with `@tool` and register them once with `add_tools`, then
ask Eve in plain English. Eve picks the tool, writes its arguments, and `run()` calls the function.

Eve reads at most 5 tools per request. The calculator has 12, so the retriever (M11) first picks
the 5 that best match the request by meaning; `considered` shows which ones Eve read.

Arithmetic is one of the domains held out of Eve's training, so the calculator shows how it does
on **new tools**: expect lower confidences, and misses when a request shares no word with
the tool's name ("plus", "times").

In [1]:
from pathlib import Path

from eve import Eve, tool

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
eve = Eve.load(
    ROOT / "artifacts/runs/joint-base/model.pt",
    retriever=ROOT / "artifacts/runs/retriever-base/model.pt",
)
eve.info["scorer"]

'Dense'

## Calculator

In [2]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@tool
def subtract(a: float, b: float) -> float:
    """Subtract the second number from the first."""
    return a - b


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


@tool
def divide(a: float, b: float) -> float:
    """Divide the first number by the second."""
    return a / b


@tool
def power(base: float, exponent: float) -> float:
    """Raise a number to a power."""
    return base**exponent


@tool
def square_root(x: float) -> float:
    """Take the square root of a number."""
    return x**0.5


@tool
def remainder(a: float, b: float) -> float:
    """The remainder after dividing the first number by the second."""
    return a % b


@tool
def percent_of(percent: float, number: float) -> float:
    """Work out a percentage of a number."""
    return percent / 100 * number


@tool
def round_number(x: float, digits: int = 0) -> float:
    """Round a number to some decimal places."""
    return round(x, digits)


@tool
def absolute_value(x: float) -> float:
    """The absolute value of a number."""
    return abs(x)


@tool
def maximum(a: float, b: float) -> float:
    """The larger of two numbers."""
    return max(a, b)


@tool
def minimum(a: float, b: float) -> float:
    """The smaller of two numbers."""
    return min(a, b)


eve.add_tools(
    add, subtract, multiply, divide, power, square_root,
    remainder, percent_of, round_number, absolute_value, maximum, minimum,
)  # fmt: skip
eve.tools

ToolBox(12 tools)

Ask Eve: it shows the intent, how sure it is, and the call it would make.

In [3]:
r = eve.respond("multiply 6 by 4")
r

<Response call 64%: multiply(a=6.0, b=4.0)>

`run()` makes the call. It refuses (with `EveError`) unless the intent is `call` and every argument is there.

In [4]:
r.run()

[24.0]

More requests, with the 5 tools the retriever showed Eve:

In [5]:
for text in [
    "add 7 and 5",
    "what is 20 plus 10",
    "subtract 3 from 10",
    "take 5 away from 12",
    "multiply 6 by 4",
    "what's 9 times 8",
    "divide 12 by 3",
    "split 100 between 4 people",
    "2 to the power of 10",
    "square root of 81",
    "what's 15 percent of 200",
    "round 3.14159 to 2 decimal places",
    "which is bigger, 7 or 12",
    "what's the weather like",
]:
    r = eve.respond(text)
    print(f"{text!r:36} {r}")
    print(f"{'':36} read {r.considered}")

'add 7 and 5'                        call 40%: add(a=7.0, b=5.0)
                                     read ['add', 'maximum', 'minimum', 'divide', 'subtract']
'what is 20 plus 10'                 unsupported 54%
                                     read ['maximum', 'minimum', 'percent_of', 'absolute_value', 'multiply']
'subtract 3 from 10'                 call 57%: subtract(a=3.0, b=10.0)
                                     read ['subtract', 'percent_of', 'add', 'multiply', 'divide']
'take 5 away from 12'                unsupported 56%
                                     read ['percent_of', 'round_number', 'maximum', 'minimum', 'multiply']
'multiply 6 by 4'                    call 64%: multiply(a=6.0, b=4.0)
                                     read ['multiply', 'add', 'subtract', 'divide', 'maximum']
"what's 9 times 8"                   unsupported 54%
                                     read ['percent_of', 'maximum', 'minimum', 'round_number', 'multiply']


'divide 12 by 3'                     call 61%: divide(a=12.0, b=3.0)
                                     read ['divide', 'add', 'power', 'maximum', 'minimum']
'split 100 between 4 people'         unsupported 53%
                                     read ['percent_of', 'maximum', 'round_number', 'minimum', 'square_root']
'2 to the power of 10'               call 58%: power(base=10.0, exponent=2.0)
                                     read ['power', 'maximum', 'minimum', 'absolute_value', 'divide']
'square root of 81'                  call 62%: square_root(x=81.0)
                                     read ['square_root', 'minimum', 'maximum', 'round_number', 'percent_of']
"what's 15 percent of 200"           call 56%: percent_of(percent=15.0, number=200.0)
                                     read ['percent_of', 'maximum', 'minimum', 'absolute_value', 'round_number']


'round 3.14159 to 2 decimal places'  call 64%: round_number(x=3.14159) invalid digits
                                     read ['round_number', 'minimum', 'square_root', 'multiply', 'maximum']
'which is bigger, 7 or 12'           unsupported 52%
                                     read ['maximum', 'minimum', 'percent_of', 'square_root', 'power']
"what's the weather like"            unsupported 54%
                                     read ['minimum', 'multiply', 'maximum', 'round_number', 'square_root']
